# Train the resistance forecaster (LightGBM)

This trains the model behind the app's `/forecast` page, with the same code and settings:
`experiments/run.py` with `experiments/configs/D3_forecaster_deploy_v7.json`.

**What it predicts.** Resistant or Susceptible for one isolate and one antibiotic, from the antibiotic,
the species and its taxonomy, and the MIC reading when there is one.

**Data.** The BV-BRC AMR export (17.6 million records), cleaned to 7.8 million rows (cleaning v7):
`supervisor_training/data/clean_v7_amr_full_norm.pkl` (1.5 GB).

**Split.** 80/20 by genome: all rows of one genome are on the same side, so the test set has only genomes the
model never saw. Target encoding is out of fold, the probabilities are calibrated (isotonic), and the threshold
keeps very major errors (resistant called susceptible) at or under 10% on the validation rows.

**What to expect.** Test AUC about 0.77 on all rows and 0.91 on the lab-tested rows
(0.774 and 0.908, the same as the app's model). On our machine the run took 4.5 minutes and 5.8 GB of RAM;
on Colab expect 10 to 20 minutes.

**Runtime.** CPU is enough, and the standard runtime (12 GB of RAM) is enough. If Colab still runs out of
memory, choose High-RAM (Runtime → Change runtime type) or set `SAMPLE_ROWS` in step 4.

**Before the first run (once).** Open the shared folder https://drive.google.com/drive/folders/1ayMzUL9K9StotIvaoViSZaVCjBnNxxyK, right-click `supervisor_training`
(or use the folder name's menu at the top) and choose **Organize → Add shortcut → My Drive**.
Colab can only read shared files through a shortcut in My Drive; nothing is copied or downloaded.

**Where the results go.** Every run gets its own folder on your Drive,
`My Drive/AMR_results/<run id>_<date and time>/`, so runs never overwrite each other.

Then run the cells from top to bottom (Runtime → Run all).

## 1. Connect Google Drive

In [ ]:
# Connect Google Drive. If this says "Mountpoint must not already contain files":
# Runtime → Disconnect and delete runtime, then Run all again.
from google.colab import drive
drive.mount('/content/drive')

import os
DATA = '/content/drive/MyDrive/supervisor_training/data'    # the shortcut to the shared folder
RESULTS_ROOT = '/content/drive/MyDrive/AMR_results'         # where every run is saved
if not os.path.isdir(DATA):
    raise FileNotFoundError(f'{DATA} not found. Add the shared folder "supervisor_training" '
                            'to My Drive as a shortcut (see the top of this notebook), '
                            'or set DATA to where it is.')
os.makedirs(RESULTS_ROOT, exist_ok=True)
!ls -lh {DATA}

## 2. Get the code

In [ ]:
# The project code, from GitHub: only the two folders training needs (a few MB, not the
# whole 1.2 GB repository). REF can be a branch or a tag.
REF = 'main'
!rm -rf /content/FYP1
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch {REF} --filter=blob:none --sparse https://github.com/hhammza/FYP1.git /content/FYP1
!git -C /content/FYP1 sparse-checkout set experiments backend
!git -C /content/FYP1 log -1 --format='code at commit %h, %cd'
!pip install -q "lightgbm>=4.0"

## 3. Link the data on Drive

In [ ]:
# The training code reads its data from experiments/cache/. Link the files on Drive there,
# so they are read straight from Drive.
CACHE = '/content/FYP1/experiments/cache'
os.makedirs(CACHE, exist_ok=True)
for name in ['clean_v7_amr_full_norm.pkl']:
    src, dst = f'{DATA}/{name}', f'{CACHE}/{name}'
    if not os.path.exists(src):
        raise FileNotFoundError(src)
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)
!ls -lL {CACHE}

## 4. The config

In [ ]:
# The config used for the model in the app. Change any field to train a different variant;
# RUN_ID names the run and its folder on Drive.
import json
from datetime import datetime
cfg = json.load(open('/content/FYP1/experiments/configs/D3_forecaster_deploy_v7.json'))
RUN_ID = 'D3_forecaster'
SAMPLE_ROWS = None   # e.g. 2_000_000 for a quicker run on less RAM; None uses every row
if SAMPLE_ROWS:
    cfg['data']['sample_rows'] = SAMPLE_ROWS
cfg['id'] = RUN_ID
RUN_DIR = f"{RESULTS_ROOT}/{RUN_ID}_{datetime.now():%Y-%m-%d_%H%M}"
os.makedirs(RUN_DIR, exist_ok=True)
MY_CONFIG = f'{RUN_DIR}/config.json'
json.dump(cfg, open(MY_CONFIG, 'w'), indent=2)
print('this run is saved to', RUN_DIR)
print(json.dumps(cfg, indent=2))

## 5. Train and evaluate

In [ ]:
# The training log is written to the run folder on Drive as it goes (train_log.txt).
%cd /content/FYP1
!python -u experiments/run.py {MY_CONFIG} 2>&1 | tee {RUN_DIR}/train_log.txt

## 6. Save the run to Drive

Each run folder on Drive holds:

| File | What it is |
|---|---|
| `config.json` | the exact settings of the run |
| `train_log.txt` | everything the training printed |
| `metrics.json` | every metric, overall, by label source and per antibiotic |
| `results_summary.csv`, `results_per_antibiotic.csv` | the two tables below, for Excel |
| `predictions.csv` | one row per test row: true label, predicted probability, genome, antibiotic |
| `model/` | the trained model and what it needs to predict |

In [ ]:
# The model, predictions and metrics, from the run into its folder on Drive
import shutil
src = f'/content/FYP1/experiments/results/{RUN_ID}'
if not os.path.exists(f'{src}/metrics.json'):
    raise RuntimeError(f'the training did not finish; see {RUN_DIR}/train_log.txt')
shutil.copytree(src, RUN_DIR, dirs_exist_ok=True)
!ls -lR {RUN_DIR}

## 7. Results

In [ ]:
import pandas as pd
m = json.load(open(f'{RUN_DIR}/metrics.json'))
cols = ['n', 'auc_roc', 'auc_pr', 'f1', 'balanced_acc', 'sensitivity', 'specificity',
        'very_major_error', 'major_error', 'threshold']
rows = {'test (all rows)': m['test']}
rows.update({f'test ({k} rows)': v for k, v in m['test_by_label_source'].items()})
summary = pd.DataFrame(rows).T[[c for c in cols if c in m['test']]]
summary.to_csv(f'{RUN_DIR}/results_summary.csv')
print(f"{m['dataset']['rows']:,} rows, {m['dataset']['genomes']:,} genomes, "
      f"{m['dataset']['antibiotics']} antibiotics, cleaning {m['dataset']['clean_version']}")
summary.round(3)

In [ ]:
# Per antibiotic, sorted by test AUC (antibiotics with few test rows are noisy)
per = pd.DataFrame(m['per_antibiotic']).set_index('group').rename_axis('antibiotic')
per.to_csv(f'{RUN_DIR}/results_per_antibiotic.csv')
per[['n', 'prevalence', 'auc_roc', 'sensitivity', 'specificity', 'very_major_error']].sort_values('auc_roc').round(3)

## 8. Use the trained model

One prediction from the saved model, the way the app makes them.

In [ ]:
%cd /content/FYP1
!python experiments/predict.py {RUN_ID} --antibiotic ciprofloxacin --species "Escherichia coli" --mic 8

## 9. All runs so far

In [ ]:
# Every run saved so far, side by side
runs = []
for d in sorted(os.listdir(RESULTS_ROOT)):
    f = f'{RESULTS_ROOT}/{d}/metrics.json'
    if os.path.exists(f):
        r = json.load(open(f))
        runs.append({'run': d, 'rows': r['dataset']['rows'], 'auc_roc': r['test']['auc_roc'],
                     'auc_roc_lab': r['test_by_label_source'].get('lab', {}).get('auc_roc'),
                     'f1': r['test']['f1'], 'very_major_error': r['test']['very_major_error'],
                     'major_error': r['test']['major_error']})
pd.DataFrame(runs).set_index('run').round(4)